# <font color="#418FDE" size="6.5" uppercase>**Multiple Features**</font>

>Last update: 20260709.
    
By the end of this Lecture, you will be able to:
- Prepare multiple scaled features for a regression model using NumPy and pandas. 
- Train a multi-feature linear regression model from scratch with gradient descent. 
- Evaluate and compare regression models using test error and visual diagnostics. 


## **1. Feature Matrix Setup**

### **1.1. Scaled Feature Matrix**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_01_01.jpg?v=1783615630" width="250">



>* Rows are observations; columns are scaled features
>* Scaling prevents large units from dominating training

>* Select clean numeric columns with pandas
>* Scale NumPy matrices for consistent model inputs

>* Learn scaling from training data only
>* Apply same scaling to all future data



In [ ]:
#@title Python Code - Scaled Feature Matrix

# Build a scaled feature matrix.
# Use pandas for readable columns.
# Use NumPy for matrix calculations.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny civil engineering dataset.
data = pd.DataFrame({
    "floor_area_sqft": [850, 1200, 1500, 1800, 2200, 2600],
    "distance_km": [1.2, 3.5, 2.1, 5.0, 4.2, 6.3],
    "age_years": [5, 12, 8, 20, 15, 30],
    "monthly_energy_kwh": [310, 420, 455, 610, 690, 820],
})

# Select multiple numeric input features.
feature_names = ["floor_area_sqft", "distance_km", "age_years"]
X_raw = data[feature_names].to_numpy(dtype=float)

# Select the regression target column.
y = data["monthly_energy_kwh"].to_numpy(dtype=float)
assert X_raw.shape == (len(y), len(feature_names))

# Learn scaling values from training data.
means = X_raw.mean(axis=0)
stds = X_raw.std(axis=0, ddof=0)

# Protect against division by zero.
stds = np.where(stds == 0, 1.0, stds)
X_scaled = (X_raw - means) / stds

# Store scaled values with readable names.
scaled_columns = [name + "_scaled" for name in feature_names]
scaled_df = pd.DataFrame(X_scaled, columns=scaled_columns)

# Show compact diagnostics only.
print("Raw feature matrix shape:", X_raw.shape)
print("Scaled feature matrix shape:", X_scaled.shape)
print("Scaled column means:", np.round(X_scaled.mean(axis=0), 3))
print("Scaled column stds:", np.round(X_scaled.std(axis=0), 3))
print("First scaled row:", np.round(X_scaled[0], 3))

# Plot raw versus scaled feature ranges.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].boxplot(X_raw, labels=feature_names)
axes[0].set_title("Raw feature ranges")
axes[0].tick_params(axis="x", rotation=25)

axes[1].boxplot(X_scaled, labels=feature_names)
axes[1].set_title("Scaled feature ranges")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()



### **1.2. Scaled Feature Matrix**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_01_02.jpg?v=1783615632" width="250">



>* Rows are observations; columns are scaled features
>* Scaling prevents large values from dominating training

>* Select clean predictor columns, excluding the target
>* Scale each feature column separately

>* Learn scaling from training data only
>* Apply the same scaling to all data



In [ ]:
#@title Python Code - Scaled Feature Matrix

# Build a scaled feature matrix.
# Use pandas and NumPy only.
# Keep training and testing separate.

# Import libraries for tabular numerical work.
import numpy as np
import pandas as pd

# Create a tiny civil engineering dataset.
data = pd.DataFrame({
    "floor_area_m2": [80, 95, 120, 150, 180, 210],
    "distance_km": [2.0, 3.5, 5.0, 7.5, 9.0, 12.0],

    "age_years": [5, 12, 20, 30, 45, 60],
    "price_kusd": [310, 330, 360, 390, 410, 430]
})

# Select multiple input feature columns.
feature_cols = ["floor_area_m2", "distance_km", "age_years"]
target_col = "price_kusd"

# Split rows before learning scaling values.
train_df = data.iloc[:4].copy()
test_df = data.iloc[4:].copy()

# Convert selected columns into NumPy matrices.
X_train = train_df[feature_cols].to_numpy(dtype=float)
X_test = test_df[feature_cols].to_numpy(dtype=float)

# Keep target values aligned with rows.
y_train = train_df[target_col].to_numpy(dtype=float)
y_test = test_df[target_col].to_numpy(dtype=float)

# Validate the feature matrix dimensions.
assert X_train.ndim == 2 and X_test.ndim == 2
assert X_train.shape[1] == len(feature_cols)

# Learn scaling values from training data only.
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

# Protect against division by zero.
train_std = np.where(train_std == 0, 1.0, train_std)
X_train_scaled = (X_train - train_mean) / train_std

# Apply the same scaling to test data.
X_test_scaled = (X_test - train_mean) / train_std
scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols)

# Show compact results for learners.
print("NumPy version:", np.__version__)
print("Training feature matrix shape:", X_train_scaled.shape)
print("Test feature matrix shape:", X_test_scaled.shape)
print("Training means after scaling:", np.round(X_train_scaled.mean(axis=0), 2))
print("Training stds after scaling:", np.round(X_train_scaled.std(axis=0), 2))
print("First scaled training row:", np.round(X_train_scaled[0], 2))
print("Target values stay unscaled here:", y_train.tolist())



### **1.3. Feature Scaling Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_01_03.jpg?v=1783615634" width="250">



>* Put different features on comparable scales
>* Prevent large units from misleading the model

>* Scaling steadies gradient descent training
>* Standardize or rescale features comparably

>* Scale each feature column separately
>* Fit scaling on training data only



In [ ]:
#@title Python Code - Feature Scaling Basics

# Demonstrate feature scaling for regression inputs.
# Use small civil engineering style data.
# Compare raw and standardized feature ranges.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny deterministic bridge dataset.
bridge_data = pd.DataFrame({
    "span_ft": [80, 120, 150, 200, 260, 310],
    "deck_width_ft": [24, 28, 30, 34, 38, 42],
    "age_years": [5, 12, 20, 35, 48, 60],
    "traffic_kvpd": [8, 15, 22, 35, 48, 60],
})

# Select multiple input features as matrix.
feature_names = ["span_ft", "deck_width_ft", "age_years", "traffic_kvpd"]
X_raw = bridge_data[feature_names].to_numpy(dtype=float)

# Validate the feature matrix shape.
rows, columns = X_raw.shape
assert rows == len(bridge_data) and columns == len(feature_names)

# Learn scaling values from training features.
feature_means = X_raw.mean(axis=0)
feature_stds = X_raw.std(axis=0, ddof=0)

# Protect against division by zero.
safe_stds = np.where(feature_stds == 0, 1.0, feature_stds)
X_scaled = (X_raw - feature_means) / safe_stds

# Store scaled features for readable inspection.
scaled_frame = pd.DataFrame(X_scaled, columns=feature_names)
raw_ranges = X_raw.max(axis=0) - X_raw.min(axis=0)
scaled_ranges = X_scaled.max(axis=0) - X_scaled.min(axis=0)

# Print compact summaries only.
print("Raw feature matrix shape:", X_raw.shape)
print("Scaled feature matrix shape:", X_scaled.shape)
print("Raw ranges:", dict(zip(feature_names, raw_ranges.round(2))))
print("Scaled ranges:", dict(zip(feature_names, scaled_ranges.round(2))))
print("Scaled means:", scaled_frame.mean().round(2).to_dict())
print("Scaled stds:", scaled_frame.std(ddof=0).round(2).to_dict())

# Plot raw versus scaled feature ranges.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(feature_names, raw_ranges, color="steelblue")
axes[0].set_title("Raw feature ranges")
axes[0].tick_params(axis="x", rotation=35)

# Show standardized ranges after scaling.
axes[1].bar(feature_names, scaled_ranges, color="darkorange")
axes[1].set_title("Scaled feature ranges")
axes[1].tick_params(axis="x", rotation=35)

# Finish with a compact visual comparison.
plt.tight_layout()
plt.show()



## **2. Gradient Descent Loop**

### **2.1. Vectorized Predictions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_02_01.jpg?v=1783615626" width="250">



>* Predict all rows using feature weights
>* Organize many features for faster modeling

>* Process all rows with current weights
>* Speed up repeated gradient descent predictions

>* Current weights generate all predictions together
>* Dataset-wide errors guide stable model updates



In [ ]:
#@title Python Code - Vectorized Predictions

# Vectorized predictions combine many feature columns.
# This example uses scaled civil data.
# Gradient descent updates all weights together.

# Import numerical and plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set deterministic random behavior.
np.random.seed(7)

# Create a tiny bridge inspection dataset.
data = pd.DataFrame({
    "span_ft": [80, 120, 150, 200, 240, 300],
    "age_years": [5, 12, 20, 35, 45, 60],
    "traffic_k": [8, 15, 22, 30, 38, 50],
    "repair_cost_k": [42, 58, 75, 105, 128, 165]
})

# Separate features and target values.
feature_cols = ["span_ft", "age_years", "traffic_k"]
X_raw = data[feature_cols].to_numpy(dtype=float)
y = data["repair_cost_k"].to_numpy(dtype=float)

# Validate the feature matrix shape.
assert X_raw.shape == (6, 3)
assert y.shape == (6,)

# Scale each feature column safely.
means = X_raw.mean(axis=0)
stds = X_raw.std(axis=0)
X_scaled = (X_raw - means) / stds

# Add an intercept column of ones.
ones = np.ones((X_scaled.shape[0], 1))
X_design = np.column_stack((ones, X_scaled))

# Initialize weights for intercept and features.
weights = np.zeros(X_design.shape[1])
learning_rate = 0.08
iterations = 300

# Run compact batch gradient descent.
for step in range(iterations):
    predictions = X_design @ weights
    errors = predictions - y
    gradient = (X_design.T @ errors) / len(y)
    weights = weights - learning_rate * gradient

# Make final vectorized predictions.
final_predictions = X_design @ weights
rmse = np.sqrt(np.mean((final_predictions - y) ** 2))

# Print concise teaching results.
print("NumPy version:", np.__version__)
print("Design matrix shape:", X_design.shape)
print("Weights shape:", weights.shape)
print("First three predictions:", np.round(final_predictions[:3], 1))
print("Training RMSE, thousand dollars:", round(rmse, 2))

# Plot actual versus predicted costs.
plt.figure(figsize=(6, 4))
plt.scatter(y, final_predictions, color="steelblue", s=70)
plt.plot([40, 170], [40, 170], color="black", linestyle="--")
plt.xlabel("Actual repair cost, thousand dollars")
plt.ylabel("Predicted repair cost, thousand dollars")
plt.title("Vectorized Predictions After Gradient Descent")
plt.grid(True, alpha=0.3)
plt.show()



### **2.2. Weight Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_02_02.jpg?v=1783615624" width="250">



>* Weights learn from prediction errors
>* Updates make small informed corrections

>* Update all feature weights together
>* Learning rate controls adjustment size

>* Bias corrects overall prediction offset
>* Repeated updates reduce errors over time



In [ ]:
#@title Python Code - Weight Updates

# Demonstrate vectorized weight updates clearly.
# Use scaled civil engineering features.
# Compare errors before and after.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set deterministic random behavior.
np.random.seed(7)

# Create a tiny bridge cost dataset.
data = pd.DataFrame({
    "span_m": [18, 25, 32, 40, 55, 70],
    "deck_width_m": [7, 8, 9, 10, 11, 12],
    "traffic_k": [4, 6, 8, 11, 14, 18],
    "cost_million": [1.8, 2.4, 3.1, 4.0, 5.4, 6.9],
})

# Select multiple input features.
feature_cols = ["span_m", "deck_width_m", "traffic_k"]
X_raw = data[feature_cols].to_numpy(dtype=float)
y = data["cost_million"].to_numpy(dtype=float)

# Validate feature and target shapes.
assert X_raw.shape[0] == y.shape[0]
assert X_raw.shape[1] == len(feature_cols)

# Scale features for stable updates.
means = X_raw.mean(axis=0)
stds = X_raw.std(axis=0)
X = (X_raw - means) / stds

# Initialize weights and bias.
weights = np.zeros(X.shape[1])
bias = 0.0
learning_rate = 0.08

# Define mean squared error.
def mse_loss(X, y, weights, bias):
    predictions = X @ weights + bias
    errors = predictions - y
    return np.mean(errors ** 2)

# Measure error before learning.
initial_loss = mse_loss(X, y, weights, bias)

# Run a compact gradient descent loop.
for step in range(120):
    predictions = X @ weights + bias
    errors = predictions - y
    weight_gradient = (2 / len(y)) * (X.T @ errors)
    bias_gradient = 2 * errors.mean()

    weights = weights - learning_rate * weight_gradient
    bias = bias - learning_rate * bias_gradient

# Measure error after learning.
final_loss = mse_loss(X, y, weights, bias)
final_predictions = X @ weights + bias

# Print concise teaching results.
print(f"NumPy version: {np.__version__}")
print(f"Initial MSE: {initial_loss:.3f}")
print(f"Final MSE: {final_loss:.3f}")
print("Learned scaled weights:")
print(pd.Series(weights, index=feature_cols).round(3))
print(f"Learned bias: {bias:.3f}")

# Plot actual versus predicted costs.
plt.figure(figsize=(6, 4))
plt.scatter(y, final_predictions, color="steelblue", s=70)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "k--")
plt.xlabel("Actual cost, million dollars")
plt.ylabel("Predicted cost, million dollars")
plt.title("Multi-feature gradient descent fit")
plt.grid(True, alpha=0.3)
plt.show()



### **2.3. Loss Tracking**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_02_03.jpg?v=1783615628" width="250">



>* Record loss during each training step
>* Check whether weight updates improve predictions

>* Loss should generally trend downward during training
>* Flat or rising loss signals problems

>* Use loss trends to diagnose training
>* Spot overfitting and compare model settings



In [ ]:
#@title Python Code - Loss Tracking

# Track loss during gradient descent.
# Use scaled civil engineering features.
# Plot learning progress clearly.

# Import numerical and plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set deterministic random seed.
rng = np.random.default_rng(42)

# Create a small housing dataset.
n_samples = 80
area_sqft = rng.normal(1800, 350, n_samples)
age_years = rng.normal(28, 10, n_samples)
transit_miles = rng.normal(2.5, 0.8, n_samples)

# Build target prices in thousands.
noise = rng.normal(0, 18, n_samples)
price_k = 55 + 0.12 * area_sqft - 1.8 * age_years - 9 * transit_miles + noise

# Store features using pandas.
data = pd.DataFrame({
    "area_sqft": area_sqft,
    "age_years": age_years,
    "transit_miles": transit_miles,
    "price_k": price_k,
})

# Select feature matrix and target.
X_raw = data[["area_sqft", "age_years", "transit_miles"]].to_numpy()
y = data["price_k"].to_numpy().reshape(-1, 1)

# Validate expected matrix dimensions.
assert X_raw.shape == (n_samples, 3)
assert y.shape == (n_samples, 1)

# Scale features for stable updates.
feature_mean = X_raw.mean(axis=0)
feature_std = X_raw.std(axis=0)
X_scaled = (X_raw - feature_mean) / feature_std

# Add intercept column to features.
ones = np.ones((n_samples, 1))
X = np.hstack([ones, X_scaled])

# Initialize weights and settings.
weights = np.zeros((X.shape[1], 1))
learning_rate = 0.08
epochs = 120
loss_history = []

# Run gradient descent silently.
for epoch in range(epochs):
    predictions = X @ weights
    errors = predictions - y
    loss = np.mean(errors ** 2)
    loss_history.append(loss)

    gradient = (2 / n_samples) * (X.T @ errors)
    weights = weights - learning_rate * gradient

# Compare early and final losses.
print(f"NumPy version: {np.__version__}")
print(f"Starting loss: {loss_history[0]:.2f}")
print(f"Final loss: {loss_history[-1]:.2f}")
print(f"Loss reduction: {loss_history[0] - loss_history[-1]:.2f}")

# Plot the tracked loss curve.
plt.figure(figsize=(7, 4))
plt.plot(loss_history, color="navy", linewidth=2)
plt.title("Loss Tracking During Gradient Descent")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



## **3. Comparing Regression Models**

### **3.1. Test Error Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_03_01.jpg?v=1783615637" width="250">



>* Test error checks performance on unseen data
>* Lower test error shows better generalization

>* Compare models using identical test conditions
>* Use test error to detect real improvement

>* Judge test error by context and risk
>* Choose practical models, not just complex ones



In [ ]:
#@title Python Code - Test Error Comparison

# Compare regression models using test error.
# Use scaled civil engineering features.
# Train simple models from scratch.

# No extra installations are required.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set deterministic randomness for reproducibility.
rng = np.random.default_rng(42)

# Create a small synthetic bridge dataset.
n = 80
span_m = rng.uniform(10, 80, n)
age_years = rng.uniform(1, 60, n)
traffic_k = rng.uniform(2, 45, n)

# Generate inspection cost in thousand dollars.
noise = rng.normal(0, 18, n)
cost_k = 40 + 1.8 * span_m + 0.9 * age_years + 1.2 * traffic_k + noise

# Store features in a readable table.
data = pd.DataFrame({
    "span_m": span_m,
    "age_years": age_years,
    "traffic_k": traffic_k,
    "cost_k": cost_k,
})

# Shuffle rows before splitting data.
data = data.sample(frac=1, random_state=7).reset_index(drop=True)

# Split into training and test sets.
train_size = int(0.75 * len(data))
train = data.iloc[:train_size].copy()
test = data.iloc[train_size:].copy()

# Define a safe feature scaling function.
def scale_features(train_df, test_df, columns):
    means = train_df[columns].mean()
    stds = train_df[columns].std(ddof=0).replace(0, 1)
    train_scaled = (train_df[columns] - means) / stds
    test_scaled = (test_df[columns] - means) / stds
    return train_scaled.to_numpy(), test_scaled.to_numpy()

# Define gradient descent for linear regression.
def fit_linear_gd(X, y, lr=0.05, epochs=1200):
    Xb = np.c_[np.ones(len(X)), X]
    weights = np.zeros(Xb.shape[1])
    for epoch in range(epochs):
        predictions = Xb @ weights
        gradient = Xb.T @ (predictions - y) / len(y)
        weights = weights - lr * gradient
    return weights

# Define prediction and mean squared error.
def predict_linear(X, weights):
    Xb = np.c_[np.ones(len(X)), X]
    return Xb @ weights

# Define a compact evaluation helper.
def evaluate_model(feature_columns):
    X_train, X_test = scale_features(train, test, feature_columns)
    y_train = train["cost_k"].to_numpy()
    y_test = test["cost_k"].to_numpy()
    assert X_train.shape[0] == y_train.shape[0]
    weights = fit_linear_gd(X_train, y_train)
    test_pred = predict_linear(X_test, weights)
    test_mse = np.mean((test_pred - y_test) ** 2)
    return test_mse, test_pred

# Compare one-feature and multi-feature models.
one_mse, one_pred = evaluate_model(["span_m"])
multi_mse, multi_pred = evaluate_model(["span_m", "age_years", "traffic_k"])

# Print a short fair comparison summary.
print(f"NumPy version: {np.__version__}")
print(f"Test rows held out: {len(test)}")
print(f"One-feature test MSE: {one_mse:.1f}")
print(f"Multi-feature test MSE: {multi_mse:.1f}")
print("Lower test MSE suggests better generalization.")

# Plot predicted versus actual test values.
plt.figure(figsize=(6, 4))
plt.scatter(test["cost_k"], one_pred, label="One feature", alpha=0.75)
plt.scatter(test["cost_k"], multi_pred, label="Multiple features", alpha=0.75)

# Add a perfect prediction reference line.
low = min(test["cost_k"].min(), one_pred.min(), multi_pred.min())
high = max(test["cost_k"].max(), one_pred.max(), multi_pred.max())
plt.plot([low, high], [low, high], "k--", label="Perfect prediction")

# Label the diagnostic plot clearly.
plt.xlabel("Actual inspection cost, thousand dollars")
plt.ylabel("Predicted inspection cost, thousand dollars")
plt.title("Test Error Comparison")
plt.legend()

# Display the single visual diagnostic.
plt.tight_layout()
plt.show()



### **3.2. Prediction Error**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_03_02.jpg?v=1783615639" width="250">



>* Compare predicted values with actual outcomes
>* Check errors for hidden patterns

>* Check where and why errors occur
>* Use errors to judge model reliability

>* Compare error patterns, not just averages
>* Choose models with acceptable mistakes



In [ ]:
#@title Python Code - Prediction Error

# Compare prediction errors across regression models.
# Use scaled civil engineering features.
# Visual diagnostics reveal systematic mistakes.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set deterministic randomness for reproducible results.
rng = np.random.default_rng(42)

# Create a small synthetic bridge dataset.
n = 80
span_m = rng.uniform(10, 80, n)
age_years = rng.uniform(1, 60, n)
traffic_k = rng.uniform(2, 45, n)

# Build a nonlinear target with noise.
noise = rng.normal(0, 8, n)
load_rating = 120 - 0.55 * span_m
load_rating += -0.35 * age_years + 0.18 * traffic_k
load_rating += -0.006 * span_m * age_years + noise

# Store features and target in pandas.
data = pd.DataFrame({
    "span_m": span_m,
    "age_years": age_years,
    "traffic_k": traffic_k,
    "load_rating": load_rating,
})

# Split rows into train and test sets.
train = data.iloc[:60].copy()
test = data.iloc[60:].copy()

# Select one-feature and multi-feature inputs.
features_one = ["span_m"]
features_multi = ["span_m", "age_years", "traffic_k"]

# Scale features using training statistics only.
mu = train[features_multi].mean()
sigma = train[features_multi].std(ddof=0)

# Validate scaling values before division.
assert np.all(sigma.to_numpy() > 0)
assert len(test) > 0 and len(train) > 0

# Prepare scaled design matrices with intercepts.
def design_matrix(frame, columns):
    scaled = (frame[columns] - mu[columns]) / sigma[columns]
    ones = np.ones((len(frame), 1))
    return np.c_[ones, scaled.to_numpy()]

# Train linear regression using gradient descent.
def fit_gradient_descent(X, y, steps=2500, lr=0.05):
    weights = np.zeros(X.shape[1])
    for step in range(steps):
        error = X @ weights - y
        gradient = X.T @ error / len(y)
        weights -= lr * gradient
    return weights

# Build arrays for both candidate models.
X1_train = design_matrix(train, features_one)
X1_test = design_matrix(test, features_one)
Xm_train = design_matrix(train, features_multi)
Xm_test = design_matrix(test, features_multi)

# Extract target arrays for training and testing.
y_train = train["load_rating"].to_numpy()
y_test = test["load_rating"].to_numpy()

# Fit both models silently from scratch.
w_one = fit_gradient_descent(X1_train, y_train)
w_multi = fit_gradient_descent(Xm_train, y_train)

# Predict test values and compute errors.
pred_one = X1_test @ w_one
pred_multi = Xm_test @ w_multi
err_one = pred_one - y_test
err_multi = pred_multi - y_test

# Summarize average and extreme prediction errors.
mae_one = np.mean(np.abs(err_one))
mae_multi = np.mean(np.abs(err_multi))
max_one = np.max(np.abs(err_one))
max_multi = np.max(np.abs(err_multi))

# Print concise comparison results.
print(f"NumPy version: {np.__version__}")
print(f"One-feature MAE: {mae_one:.2f} rating points")
print(f"Multi-feature MAE: {mae_multi:.2f} rating points")
print(f"One-feature largest error: {max_one:.2f}")
print(f"Multi-feature largest error: {max_multi:.2f}")

# Plot prediction error against actual bridge rating.
plt.figure(figsize=(7, 4))
plt.axhline(0, color="black", linewidth=1)
plt.scatter(y_test, err_one, label="One feature", alpha=0.8)
plt.scatter(y_test, err_multi, label="Multiple features", alpha=0.8)
plt.xlabel("Actual load rating")
plt.ylabel("Prediction error")
plt.title("Prediction Error on Test Bridges")
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Visual Model Diagnostics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_04/Lecture_B/image_03_03.jpg?v=1783615641" width="250">



>* Compare predictions with actual values visually
>* Spot uneven errors across prediction ranges

>* Residual plots reveal model error patterns
>* Patterns suggest missing nonlinear or subgroup effects

>* Plots reveal hidden model behavior differences
>* Combine visuals with test error decisions



In [ ]:
#@title Python Code - Visual Model Diagnostics

# Visual diagnostics reveal model behavior.
# Residual plots show hidden patterns.
# Test error needs visual context.

# Import numerical and plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set deterministic random seed.
rng = np.random.default_rng(42)

# Create a small housing dataset.
n = 80
area = rng.uniform(500, 2500, n)
distance = rng.uniform(0.5, 12.0, n)

# Add nonlinear high-area price effect.
bonus = 0.00008 * np.maximum(area - 1700, 0) ** 2
noise = rng.normal(0, 18000, n)
price = 60000 + 145 * area - 4500 * distance + bonus + noise

# Store features in a dataframe.
data = pd.DataFrame({"area_sqft": area, "distance_miles": distance, "price": price})

# Split data into train and test sets.
train = data.iloc[:60].copy()
test = data.iloc[60:].copy()

# Scale features using training statistics.
feature_cols = ["area_sqft", "distance_miles"]
means = train[feature_cols].mean()
stds = train[feature_cols].std(ddof=0)

# Validate scaling values before division.
assert len(test) > 0 and np.all(stds.to_numpy() > 0)
X_train = ((train[feature_cols] - means) / stds).to_numpy()
X_test = ((test[feature_cols] - means) / stds).to_numpy()

# Add intercept columns for linear models.
ones_train = np.ones((X_train.shape[0], 1))
ones_test = np.ones((X_test.shape[0], 1))
X_train_basic = np.c_[ones_train, X_train]
X_test_basic = np.c_[ones_test, X_test]

# Add a squared area feature.
area_train_sq = X_train[:, [0]] ** 2
area_test_sq = X_test[:, [0]] ** 2
X_train_curve = np.c_[X_train_basic, area_train_sq]
X_test_curve = np.c_[X_test_basic, area_test_sq]

# Prepare target arrays for training.
y_train = train["price"].to_numpy()
y_test = test["price"].to_numpy()

# Define compact gradient descent training.
def fit_linear(X, y, lr=0.03, steps=2500):
    weights = np.zeros(X.shape[1])
    scale = max(float(np.std(y)), 1.0)
    y_scaled = y / scale

    for step in range(steps):
        errors = X @ weights - y_scaled
        gradient = X.T @ errors / len(y)
        weights -= lr * gradient

    return weights * scale

# Train two comparable regression models.
w_basic = fit_linear(X_train_basic, y_train)
w_curve = fit_linear(X_train_curve, y_train)

# Predict test prices for both models.
pred_basic = X_test_basic @ w_basic
pred_curve = X_test_curve @ w_curve

# Compute test root mean squared errors.
rmse_basic = np.sqrt(np.mean((pred_basic - y_test) ** 2))
rmse_curve = np.sqrt(np.mean((pred_curve - y_test) ** 2))

# Print concise numerical comparison.
print(f"NumPy version: {np.__version__}")
print(f"Basic model test RMSE: ${rmse_basic:,.0f}")
print(f"Curved model test RMSE: ${rmse_curve:,.0f}")
print("Use the plots to compare error patterns.")

# Create residuals for visual diagnostics.
res_basic = pred_basic - y_test
res_curve = pred_curve - y_test

# Build one figure with two diagnostic panels.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Plot predicted versus actual values.
axes[0].scatter(y_test, pred_basic, label="Basic", alpha=0.8)
axes[0].scatter(y_test, pred_curve, label="Curved", alpha=0.8)
lims = [y_test.min() * 0.95, y_test.max() * 1.05]
axes[0].plot(lims, lims, "k--", label="Perfect")

# Label the prediction diagnostic plot.
axes[0].set_title("Predicted versus actual price")
axes[0].set_xlabel("Actual price, dollars")
axes[0].set_ylabel("Predicted price, dollars")
axes[0].legend()

# Plot residuals against predicted values.
axes[1].scatter(pred_basic, res_basic, label="Basic", alpha=0.8)
axes[1].scatter(pred_curve, res_curve, label="Curved", alpha=0.8)
axes[1].axhline(0, color="black", linestyle="--")

# Label the residual diagnostic plot.
axes[1].set_title("Residuals versus predicted price")
axes[1].set_xlabel("Predicted price, dollars")
axes[1].set_ylabel("Residual, predicted minus actual")
axes[1].legend()

# Display the single diagnostic figure.
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Multiple Features**</font>


In this lecture, you learned to:
- Prepare multiple scaled features for a regression model using NumPy and pandas. 
- Train a multi-feature linear regression model from scratch with gradient descent. 
- Evaluate and compare regression models using test error and visual diagnostics. 

In the next Module (Module 5), we will go over 'None'